In [ ]:
"""
NoBroker Property Data Extractor --
---------------------------------
Reads input.csv (columns: id, title, link, date, savedAt)
Extracts property data from each NoBroker listing URL
Saves to final_excel.xlsx (skips already-extracted records)

Requirements (install once):
    pip install selenium openpyxl pandas bs4 lxml requests
"""

# ── Auto-install missing packages (same pattern as your working code) ───────────
import subprocess
import sys

def import_or_install(package):
    try:
        __import__(package)
    except ImportError:
        print(f"{package} not installed. Installing...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
        print(f"{package} has been installed.")
    else:
        print(f"{package} is already installed.")

import_or_install('selenium')
import_or_install('openpyxl')
import_or_install('pandas')
import_or_install('bs4')
import_or_install('lxml')

# ── Imports ──────────────────────────────────────────────────────────────────────
import os
import sys
import time
import random
import logging
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ── Config ────────────────────────────────────────────────────────────────────────
INPUT_CSV    = "input.csv"
OUTPUT_EXCEL = "final_excel.xlsx"
LOGIN_WAIT   = 40    # seconds to wait for manual login
PAGE_WAIT    = 15    # seconds to wait for each listing page to load

# ── Columns — same 33 fields as the JS bookmarklet ───────────────────────────────
COLUMNS = [
    "id", "url", "status", "phone", "visitDate",
    "title", "location", "rent", "maintenance", "area", "deposit",
    "bedrooms", "propertyType", "preferredTenant", "possession",
    "parking", "buildingAge", "balcony", "postedOn",
    "furnishingStatus", "facing", "waterSupply", "floor", "bathroom",
    "petAllowed", "nonVegAllowed", "gatedSecurity",
    "latitude", "longitude",
    "uniqueViews", "shortlists", "contacted",
    "comments"
]

# ── Logging ───────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("nobroker_scraper.log", encoding="utf-8")
    ]
)
log = logging.getLogger(__name__)

# ── JS extraction — mirrors the bookmarklet exactly ───────────────────────────────
EXTRACT_JS = """
function extractIdFromUrl(url) {
    try {
        var match = url.match(/\\/property\\/[^\\/]+\\/([a-f0-9]+)\\/detail/i);
        if (match && match[1]) return match[1];
        var parts = url.split('/');
        var detailIndex = parts.indexOf('detail');
        if (detailIndex > 0) return parts[detailIndex - 1];
        return '';
    } catch(e) { return ''; }
}

function safeExtract(selector, attribute) {
    try {
        attribute = attribute || 'textContent';
        var el = document.querySelector(selector);
        if (!el) return '';
        if (attribute === 'textContent') return el.textContent.trim().replace(/\\s+/g, ' ');
        if (attribute === 'content')     return el.getAttribute('content') || '';
        return el.getAttribute(attribute) || '';
    } catch(e) { return ''; }
}

function extractById(id) {
    try {
        var el = document.getElementById(id);
        return el ? el.textContent.trim().replace(/\\s+/g, ' ') : '';
    } catch(e) { return ''; }
}

function extractOverviewItem(title) {
    try {
        var items = document.querySelectorAll('.nb__3ocPe');
        for (var i = 0; i < items.length; i++) {
            var titleEl = items[i].querySelector('.nb__1IoiM');
            if (titleEl && titleEl.textContent.includes(title)) {
                var valueEl = items[i].querySelector('.font-semi-bold');
                return valueEl ? valueEl.textContent.trim() : '';
            }
        }
        return '';
    } catch(e) { return ''; }
}

var data = {
    id:               extractIdFromUrl(window.location.href),
    url:              window.location.href,
    status:           'Not started',
    phone:            '',
    visitDate:        '',
    title:            safeExtract('h1.nb__s_YQN'),
    location:         safeExtract('h5.nb__16pur'),
    rent:             safeExtract('#rent-maintenance span[itemprop="value"] > div > div > span') ||
                      safeExtract('#rent-maintenance .flex.gap-1 > span') ||
                      safeExtract('#rent-maintenance .nb__3h7Fo > span[itemprop="value"]'),
    maintenance:      safeExtract('#rent-maintenance .text-14.font-semibold.ml-1'),
    area:             safeExtract('#square-ft'),
    deposit:          safeExtract('#emi span[itemprop="value"]') || safeExtract('#emi'),
    bedrooms:         extractById('details-summary-typeDesc'),
    propertyType:     extractById('details-summary-buildingType'),
    preferredTenant:  extractById('details-summary-leaseType'),
    possession:       extractById('details-summary-availableFrom'),
    parking:          extractById('details-summary-parkingDesc'),
    buildingAge:      extractById('details-summary-propertyAge'),
    balcony:          extractById('details-summary-balconies'),
    postedOn:         extractById('details-summary-lastUpdateDate'),
    furnishingStatus: extractOverviewItem('Furnishing Status'),
    facing:           extractOverviewItem('Facing'),
    waterSupply:      extractOverviewItem('Water Supply'),
    floor:            extractOverviewItem('Floor'),
    bathroom:         extractOverviewItem('Bathroom'),
    petAllowed:       extractOverviewItem('Pet Allowed'),
    nonVegAllowed:    extractOverviewItem('Non-Veg Allowed'),
    gatedSecurity:    extractOverviewItem('Gated Security'),
    latitude:         safeExtract('meta[itemprop="latitude"]',  'content'),
    longitude:        safeExtract('meta[itemprop="longitude"]', 'content'),
    uniqueViews:      '',
    shortlists:       '',
    contacted:        '',
    comments:         ''
};

try {
    var actDivs = document.querySelectorAll('.nb__3PMUu');
    if (actDivs.length >= 1) data.uniqueViews = ((actDivs[0].querySelector('.nb__32dA0') || {}).textContent || '').trim();
    if (actDivs.length >= 2) data.shortlists  = ((actDivs[1].querySelector('.nb__32dA0') || {}).textContent || '').trim();
    if (actDivs.length >= 3) data.contacted   = ((actDivs[2].querySelector('.nb__32dA0') || {}).textContent || '').trim();
} catch(e) {}

return data;
"""


# ── Random wait (same as your working code) ───────────────────────────────────────
def wait(num=5):
    t = random.randint(1, num)
    time.sleep(t)


# ── Excel helpers ─────────────────────────────────────────────────────────────────
def load_or_create_excel(path):
    if os.path.exists(path):
        log.info(f"Loading existing Excel: {path}")
        df = pd.read_excel(path, dtype=str).fillna("")
        for col in COLUMNS:           # add any missing columns
            if col not in df.columns:
                df[col] = ""
        return df[COLUMNS]
    else:
        log.info(f"final_excel.xlsx not found — creating new file: {path}")
        df = pd.DataFrame(columns=COLUMNS)
        df.to_excel(path, index=False)
        return df


def save_excel(df, path):
    df.to_excel(path, index=False)
    log.info(f"  💾  Saved {len(df)} row(s) → {path}")


# ── Wait for page load ────────────────────────────────────────────────────────────
def wait_for_page(driver, timeout=PAGE_WAIT):
    WebDriverWait(driver, timeout).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    time.sleep(3)   # extra buffer for JS-heavy content


# ── Main ──────────────────────────────────────────────────────────────────────────
def main():

    # 1. Load input CSV
    if not os.path.exists(INPUT_CSV):
        log.error(f"Input file not found: {INPUT_CSV}")
        sys.exit(1)

    input_df = pd.read_csv(INPUT_CSV, dtype=str).fillna("")
    log.info(f"Input CSV: {len(input_df)} row(s) loaded")

    if "link" not in input_df.columns:
        log.error("Column 'link' not found in input.csv")
        sys.exit(1)

    # 2. Load / create output Excel
    out_df = load_or_create_excel(OUTPUT_EXCEL)
    existing_urls = set(out_df["url"].dropna().str.strip())
    log.info(f"Already in Excel: {len(existing_urls)} record(s)")

    # 3. Filter out already-done rows
    pending = [
        row for _, row in input_df.iterrows()
        if str(row.get("link", "")).strip() not in existing_urls
    ]
    log.info(f"Skipping : {len(input_df) - len(pending)} already-extracted")
    log.info(f"Pending  : {len(pending)} to extract")

    if not pending:
        log.info("All records already extracted. Nothing to do.")
        return

    # 4. Launch Chrome — same as your working Naukri code
    driver = webdriver.Chrome()
    log.info("Chrome launched.")

    try:
        driver.get("https://www.nobroker.in")
        log.info(f"⏳  Please log in manually in the browser window.")
        log.info(f"   You have {LOGIN_WAIT} seconds…")

        for remaining in range(LOGIN_WAIT, 0, -5):
            log.info(f"   {remaining}s remaining to log in…")
            time.sleep(5)

        log.info("Login wait done. Starting property extraction…")

        # 5. Iterate and extract
        total = len(pending)
        for idx, row in enumerate(pending, start=1):
            url = str(row.get("link", "")).strip()
            log.info(f"[{idx}/{total}]  {url}")

            try:
                driver.get(url)
                wait_for_page(driver)

                data = driver.execute_script(EXTRACT_JS)
                data["url"] = url   # preserve original input URL

                log.info(
                    f"  ✔  title='{data.get('title', '')}' | "
                    f"rent='{data.get('rent', '')}' | "
                    f"location='{data.get('location', '')}'"
                )
                record = {col: str(data.get(col, "") or "").strip() for col in COLUMNS}

            except Exception as e:
                log.error(f"  ✗  Failed: {e}")
                record = {col: "" for col in COLUMNS}
                record["url"]    = url
                record["status"] = "ERROR"

            # Append and save immediately after every record (crash-safe)
            out_df = pd.concat(
                [out_df, pd.DataFrame([record], columns=COLUMNS)],
                ignore_index=True
            )
            save_excel(out_df, OUTPUT_EXCEL)

            wait(3)   # polite random delay between requests

        log.info(f"✅  Done! {total} record(s) processed.")
        log.info(f"📄  Output: {OUTPUT_EXCEL} ({len(out_df)} total rows)")

    finally:
        driver.quit()
        log.info("Browser closed.")


if __name__ == "__main__":
    main()